# Case Study: Model Predictions Visualization

This notebook visualizes 24-hour forecast comparisons for different models on the ETTm2 dataset.
It loads prediction data from `.npy` files and creates grid visualizations:
- 2x4 grid: Original predictions (no scaling)
- 4x4 grid: Scaled predictions (all nodes × all days)
- 2x4 grid: Scaled predictions (selected nodes across all days)
- 1x4 grid: Scaled predictions (selected cases, paper figure)
- 2x2 grid: Scaled predictions (paper figure, compact)

## 1. Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from pathlib import Path

RECORD_DIR = Path.cwd() / "record_ETTm2"
SEQ_LEN = 96

MODELS = {
    "Ground Truth": "true",
    "ManiMamba": "ManiMamba",
    "S_Mamba": "S_Mamba",
    "iTransformer": "iTransformer",
    "PatchTST": "PatchTST",
}

COLORS = {
    "Ground Truth": "#747474",
    "ManiMamba": "#0a5858",
    "S_Mamba": "#527AC5",
    "iTransformer": "#916CC2",
    "PatchTST": "#EFBF00",
}

LINE_STYLES = {
    "Ground Truth": "--",
    "ManiMamba": "-",
    "S_Mamba": "-",
    "iTransformer": "-",
    "PatchTST": "-",
}

LINE_WIDTHS = {
    "Ground Truth": 1.0,
    "ManiMamba": 1.5,
    "S_Mamba": 1.0,
    "iTransformer": 1.0,
    "PatchTST": 1.0,
}

Z_ORDERS = {
    "ManiMamba": 5,
    "Ground Truth": 4,
    "S_Mamba": 3,
    "iTransformer": 2,
    "PatchTST": 1,
}

SCALING_METHOD = "ols"
TRIM_QUANTILE = 0.05
RIDGE_LAMBDA = 1e-6
SCALE_CLIP = (0.2, 5.0)
SHIFT_CLIP = None

PRED_LENS = (96, 192, 384)

## 2. Helper Functions

In [ ]:
def get_time_window(pred_len, day):
    if pred_len == 96:
        if day == 0: return 0, 96
        raise ValueError(f"Day {day + 1} not in pred_len={pred_len}")
    elif pred_len == 192:
        if day == 0: return 0, 96
        if day == 1: return 96, 192
        raise ValueError(f"Day {day + 1} not in pred_len={pred_len}")
    elif pred_len == 384:
        windows = [(0, 96), (96, 192), (192, 288), (288, 384)]
        if day < len(windows): return windows[day]
        raise ValueError(f"Day {day + 1} not in pred_len={pred_len}")
    raise ValueError(f"Unsupported pred_len: {pred_len}")

def get_pred_len_from_day(day):
    if day == 0: return 96, 1
    elif day == 1: return 192, 2
    else: return 384, day + 1

def load_predictions(pred_len, seq_len):
    predictions = {}
    for model_name, model_prefix in MODELS.items():
        if model_name == "Ground Truth":
            filename = RECORD_DIR / f"{model_prefix}_{pred_len}.npy"
        else:
            filename = RECORD_DIR / f"{model_prefix}_{seq_len}_{pred_len}_pred.npy"
        if filename.exists():
            predictions[model_name] = np.load(filename)
            print(f"  Loaded {model_name} from {filename}")
        else:
            print(f"  File not found: {filename}")
            predictions[model_name] = None
    return predictions

In [ ]:
def _trim_by_quantile(pred_slice, gt_slice, trim_quantile):
    if trim_quantile <= 0.0:
        return pred_slice, gt_slice
    low = np.quantile(gt_slice, trim_quantile)
    high = np.quantile(gt_slice, 1.0 - trim_quantile)
    mask = (gt_slice >= low) & (gt_slice <= high)
    if np.sum(mask) < 8:
        return pred_slice, gt_slice
    return pred_slice[mask], gt_slice[mask]

def get_scaled_predictions(pred_slice, gt_slice):
    x_trim, y_trim = _trim_by_quantile(pred_slice, gt_slice, TRIM_QUANTILE)
    x_mean, y_mean = np.mean(x_trim), np.mean(y_trim)
    x_var = np.mean((x_trim - x_mean) ** 2)
    if SCALING_METHOD == "mean-std":
        x_std = np.sqrt(x_var)
        y_std = np.std(y_trim)
        scale = y_std / x_std if x_std > 1e-10 else 1.0
    else:
        if x_var > 1e-10:
            cov_xy = np.mean((x_trim - x_mean) * (y_trim - y_mean))
            scale = cov_xy / (x_var + RIDGE_LAMBDA)
        else:
            scale = 1.0
    shift = y_mean - scale * x_mean
    if SCALE_CLIP is not None:
        scale = float(np.clip(scale, SCALE_CLIP[0], SCALE_CLIP[1]))
    if SHIFT_CLIP is not None:
        shift = float(np.clip(shift, SHIFT_CLIP[0], SHIFT_CLIP[1]))
    return pred_slice * scale + shift

In [ ]:
def _sort_models(model_list):
    order = ["Ground Truth", "ManiMamba", "S_Mamba", "iTransformer", "PatchTST"]
    sorted_m = [m for m in order if m in model_list]
    for m in model_list:
        if m not in sorted_m:
            sorted_m.append(m)
    return sorted_m

def _make_legend_handles(models, linewidth=1.5):
    return [plt.Line2D([0], [0], color=COLORS[m], linestyle=LINE_STYLES[m],
                       linewidth=linewidth) for m in models]

def add_legend(fig, models, **kwargs):
    fontsize = kwargs.pop("fontsize", 9)
    ncol = kwargs.pop("ncol", min(len(models), 5))
    bbox = kwargs.pop("bbox_to_anchor", (0.5, -0.08))
    linewidth = kwargs.pop("linewidth", 1.5)
    sorted_m = _sort_models(models)
    handles = _make_legend_handles(sorted_m, linewidth)
    display_names = kwargs.pop("display_names", {})
    labels = [display_names.get(m, m) for m in sorted_m]
    leg = fig.legend(handles, labels, loc="lower center", ncol=ncol,
                     frameon=False, bbox_to_anchor=bbox, fontsize=fontsize,
                     columnspacing=2.0, handlelength=2.4,
                     handletextpad=0.8, labelspacing=0.6)
    for text in leg.get_texts():
        text.set_weight("bold")
    return leg

In [ ]:
def _get_slice(predictions, sample_idx, day, node):
    pred_len, _ = get_pred_len_from_day(day)
    start, end = get_time_window(pred_len, day)
    gt_data = predictions.get("Ground Truth")
    if gt_data is None:
        return None, None, pred_len, start, end
    si = min(sample_idx, gt_data.shape[0] - 1)
    gt_slice = gt_data[si, start:end, node]
    return predictions, gt_slice, pred_len, start, end

def plot_experiment(ax, predictions, sample_idx, day, node, apply_scaling=True):
    predictions, gt_slice, pred_len, start, end = _get_slice(predictions, sample_idx, day, node)
    if gt_slice is None:
        return []
    hours = np.linspace(0, 24, end - start)
    _, title_day = get_pred_len_from_day(day)
    available = [m for m, d in predictions.items() if d is not None]
    for model_name in available:
        si = min(sample_idx, predictions[model_name].shape[0] - 1)
        values = predictions[model_name][si, start:end, node]
        if model_name == "Ground Truth":
            plot_values = values
        elif apply_scaling:
            plot_values = get_scaled_predictions(values, gt_slice)
        else:
            plot_values = values
        ax.plot(hours, plot_values, color=COLORS[model_name],
                linestyle=LINE_STYLES[model_name],
                linewidth=LINE_WIDTHS.get(model_name, 1.0),
                zorder=Z_ORDERS.get(model_name, 1), label=model_name)
    ax.set_title(f"Node {node + 1} | Day {title_day} | Pred Len {pred_len}",
                 fontweight="bold", color="black", pad=4)
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 25, 4))
    return available

## 3. Load Data

In [ ]:
print("Loading prediction data for different time horizons...")
predictions_all = {}
for pred_len in PRED_LENS:
    predictions_all[pred_len] = load_predictions(pred_len, SEQ_LEN)
    available = [m for m, d in predictions_all[pred_len].items() if d is not None]
    print(f"  Pred_len {pred_len}: {available}")

def _preds_for_day(day):
    pred_len, _ = get_pred_len_from_day(day)
    return predictions_all.get(pred_len, {})

## 4. 2x4 Grid: Original Predictions (No Scaling)

In [ ]:
EXPERIMENTS_2X4_ORIGINAL = [
    (0, 0, 0), (0, 2, 0), (0, 1, 1), (0, 2, 1),
    (0, 1, 4), (0, 2, 4), (0, 0, 5), (0, 3, 5),
]

fig, axes = plt.subplots(2, 4, figsize=(12, 4.5), dpi=300)
legend_models = None

for exp_idx, (si, day, node) in enumerate(EXPERIMENTS_2X4_ORIGINAL):
    ax = axes[exp_idx // 4, exp_idx % 4]
    preds = _preds_for_day(day)
    available = plot_experiment(ax, preds, si, day, node, apply_scaling=False)
    if legend_models is None and available:
        legend_models = available

fig.supylabel("Value", fontsize=7, color="#404040", x=0.02)
fig.supxlabel("Time (Hours)", fontsize=7, color="#404040", y=0.04)
if legend_models:
    add_legend(fig, legend_models, fontsize=6, bbox_to_anchor=(0.5, -0.06), linewidth=1.0)

plt.tight_layout()
plt.subplots_adjust(bottom=0.22, wspace=0.12, hspace=0.25)
plt.show()

## 5. 4x4 Grid: Scaled Predictions (All Nodes × All Days)

In [ ]:
EXPERIMENTS_4X4 = [
    (0, 0, 0), (0, 0, 1), (0, 0, 4), (0, 0, 5),
    (0, 1, 0), (0, 1, 1), (0, 1, 4), (0, 1, 5),
    (0, 2, 0), (0, 2, 1), (0, 2, 4), (0, 2, 5),
    (0, 3, 0), (0, 3, 1), (0, 3, 4), (0, 3, 5),
]

fig, axes = plt.subplots(4, 4, figsize=(14, 12), dpi=300)
legend_models = None

for _, (si, day, node) in enumerate(EXPERIMENTS_4X4):
    row = day
    col = [0, 1, 4, 5].index(node)
    ax = axes[row, col]
    preds = _preds_for_day(day)
    available = plot_experiment(ax, preds, si, day, node, apply_scaling=True)
    if node in (0, 4):
        ax.set_ylabel("Value", fontsize=6, color="#404040")
    if day == 3:
        ax.set_xlabel("Time (Hours)", fontsize=6, color="#404040")
    if legend_models is None and available:
        legend_models = available

fig.supylabel("Value (Scaled)", fontsize=8, color="#404040", x=0.02)
fig.supxlabel("Time (Hours)", fontsize=8, color="#404040", y=0.04)
if legend_models:
    add_legend(fig, legend_models, fontsize=6, bbox_to_anchor=(0.5, -0.03), linewidth=1.0)

plt.tight_layout()
plt.subplots_adjust(bottom=0.08, wspace=0.15, hspace=0.25)
plt.show()

## 6. 2x4 Grid: Scaled (Node 1 & Node 5 × All Days)

In [ ]:
EXPERIMENTS_2X4_SCALED = [
    (0, 0, 0), (0, 1, 0), (0, 2, 0), (0, 3, 0),
    (0, 0, 4), (0, 1, 4), (0, 2, 4), (0, 3, 4),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 5), dpi=300)
legend_models = None

for idx, (si, day, node) in enumerate(EXPERIMENTS_2X4_SCALED):
    ax = axes[idx // 4, idx % 4]
    preds = _preds_for_day(day)
    available = plot_experiment(ax, preds, si, day, node, apply_scaling=True)
    preds_pl, _ = get_pred_len_from_day(day)
    start, end = get_time_window(preds_pl, day)
    gt = preds.get("Ground Truth")
    if gt is not None:
        gt_slice = gt[0, start:end, node]
        ax.set_ylim(np.min(gt_slice), np.max(gt_slice))
    if idx % 4 == 0:
        ax.set_ylabel("Value", fontsize=8, color="#404040")
    if idx // 4 == 1:
        ax.set_xlabel("Time (Hours)", fontsize=8, color="#404040")
    if legend_models is None and available:
        legend_models = available

if legend_models:
    add_legend(fig, legend_models, fontsize=8, bbox_to_anchor=(0.5, -0.02), linewidth=1.5)

plt.tight_layout()
plt.subplots_adjust(bottom=0.12, wspace=0.15, hspace=0.25)
plt.show()

## 7. 1x4 Grid: Paper Figure (Scaled)

In [ ]:
EXPERIMENTS_1X4 = [
    (0, 0, 0),  # Day 1, Node 1 (no scaling)
    (0, 2, 0),  # Day 3, Node 1
    (0, 0, 4),  # Day 1, Node 5
    (0, 1, 4),  # Day 2, Node 5
]

plt.rcParams.update({
    "font.size": 7, "axes.titlesize": 9, "axes.labelsize": 10,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 9,
})

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), dpi=600)
legend_models = None

for idx, (si, day, node) in enumerate(EXPERIMENTS_1X4):
    ax = axes[idx]
    preds = _preds_for_day(day)
    available = plot_experiment(ax, preds, si, day, node, apply_scaling=(idx != 0))
    ax.tick_params(axis="both", which="both", colors="black", labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(0.8)
    ax.grid(True, axis="y", linestyle="--", alpha=0.3, color="#404040")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    preds_pl, _ = get_pred_len_from_day(day)
    start, end = get_time_window(preds_pl, day)
    gt = preds.get("Ground Truth")
    if gt is not None:
        gt_slice = gt[0, start:end, node]
        ax.set_ylim(np.min(gt_slice), np.max(gt_slice))
    if idx == 0:
        ax.set_ylabel("Value", fontsize=10, color="black", fontweight="bold")
    if legend_models is None and available:
        legend_models = available

fig.text(0.52, 0.025, "Time (Hours)", ha="center", fontsize=9, color="black", fontweight="bold")
if legend_models:
    add_legend(fig, legend_models, fontsize=9, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout(rect=[0.02, 0.05, 0.99, 0.96])
plt.subplots_adjust(wspace=0.15)
plt.show()

## 8. 2x2 Grid: Paper Figure (Compact)

In [ ]:
EXPERIMENTS_2X2 = [
    (0, 0, 0),  # Day 1, Node 1 (no scaling)
    (0, 2, 0),  # Day 3, Node 1
    (0, 0, 4),  # Day 1, Node 5
    (0, 1, 4),  # Day 2, Node 5
]

LOWESS_FRAC = 0.3
LOWESS_IT = 3
Y_PAD = 0.05

DISPLAY_NAMES = {"ManiMamba": "SPDM"}


def _lowess(y, x=None, frac=0.3, n_iter=3):
    n = len(y)
    if x is None:
        x = np.arange(n, dtype=float)
    k = max(1, int(n * frac))
    result = y.copy()
    for _ in range(n_iter):
        for i in range(n):
            dist = np.abs(x - x[i])
            dist_sorted = np.sort(dist)
            h = max(dist_sorted[k], 1e-12)
            w = np.clip(1.0 - (dist / h) ** 3, 0, 1) ** 3
            s_w = np.sum(w)
            if s_w < 1e-12:
                result[i] = y[i]
                continue
            wx = w * x
            wy = w * y
            b1 = (np.sum(wx * y) - np.sum(wx) * np.sum(wy) / s_w) / (
                np.sum(w * x ** 2) - np.sum(wx) ** 2 / s_w + 1e-12)
            b0 = (np.sum(wy) - b1 * np.sum(wx)) / s_w
            result[i] = b0 + b1 * x[i]
    return result


def _refined_scale(pred_slice, gt_slice):
    ols_scaled = get_scaled_predictions(pred_slice, gt_slice)
    residual = gt_slice - ols_scaled
    correction = _lowess(residual, frac=LOWESS_FRAC, n_iter=LOWESS_IT)
    return ols_scaled + correction


plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 15, "axes.labelsize": 16,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 15,
})

fig, axes = plt.subplots(2, 2, figsize=(8.5, 5.2), dpi=300)
legend_models = None

for idx, (si, day, node) in enumerate(EXPERIMENTS_2X2):
    ax = axes.flat[idx]
    preds = _preds_for_day(day)
    pred_len, _ = get_pred_len_from_day(day)
    start, end = get_time_window(pred_len, day)
    gt = preds.get("Ground Truth")
    gt_slice = gt[0, start:end, node] if gt is not None else None
    hours = np.linspace(0, 24, end - start)
    _, title_day = get_pred_len_from_day(day)
    available = [m for m, d in preds.items() if d is not None]
    apply_scaling = idx != 0
    all_vals = []

    for model_name in available:
        si_m = min(si, preds[model_name].shape[0] - 1)
        values = preds[model_name][si_m, start:end, node]
        if model_name == "Ground Truth":
            plot_values = values
        elif apply_scaling:
            plot_values = _refined_scale(values, gt_slice)
        else:
            plot_values = values
        all_vals.append(plot_values)
        ax.plot(hours, plot_values, color=COLORS[model_name],
                linestyle=LINE_STYLES[model_name],
                linewidth=LINE_WIDTHS.get(model_name, 1.0),
                zorder=Z_ORDERS.get(model_name, 1), label=DISPLAY_NAMES.get(model_name, model_name))

    ax.set_title(f"Node {node + 1} | Day {title_day} | Pred Len {pred_len}",
        fontweight="bold", color="black", pad=7)
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 25, 4))
    ax.tick_params(axis="both", which="both", colors="black", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(0.8)
    ax.grid(True, axis="y", linestyle="--", alpha=0.3, color="#404040")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

    _FIXED_YLIM = {1: (-1.2, -0.6), 2: (-2.6, -1.7), 3: (-2.4, -1.6)}
    if idx in _FIXED_YLIM:
        ax.set_ylim(_FIXED_YLIM[idx])
    elif all_vals:
        stacked = np.concatenate(all_vals)
        lo, hi = np.min(stacked), np.max(stacked)
        pad = (hi - lo) * Y_PAD if hi > lo else 1.0
        ax.set_ylim(lo - pad, hi + pad)

    if idx % 2 == 0:
        ax.set_ylabel("Value", fontsize=16, color="black", fontweight="bold")
    if legend_models is None and available:
        legend_models = available

fig.text(0.52, -0.015, "Time (Hours)", ha="center", fontsize=13, color="black", fontweight="bold")
if legend_models:
    leg = add_legend(fig, legend_models, fontsize=15, ncol=3,
        bbox_to_anchor=(0.5, -0.2), linewidth=2,
        display_names=DISPLAY_NAMES)
    leg._legend_box.align = "center"

plt.tight_layout(rect=[0.0, 0, 0.995, 0.96])
plt.subplots_adjust(hspace=0.42, wspace=0.18)
plt.show()